In [ ]:
import os
import tensorflow as tf

!pip install -q kaggle
os.environ['KAGGLE_USERNAME'] = "YOUR_KAGGLE_USERNAME"
os.environ['KAGGLE_KEY'] = "YOUR_KAGGLE_KEY"

print(">>> Downloading dataset...")
!kaggle datasets download -d cjinny/mura-v11 -o
!unzip -q mura-v11.zip -d /content/MURA_DATASET

if os.paths.exists("mura-v11.zip"):
    os.remove("mura-v11.zip")

print(">>> Dataset Ready!")

>>> Downloading dataset...
Dataset URL: https://www.kaggle.com/datasets/cjinny/mura-v11
License(s): unknown
100% 3.14G/3.14G [02:29<00:00, 22.5MB/s]

>>> Dataset Ready! Folders: ['MURA-v1.1']


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

train_df = pd.read_csv('MURA_DATASET/MURA-v1.1/train_image_paths.csv', names=['path'])
valid_df = pd.read_csv('MURA_DATASET/MURA-v1.1/valid_image_paths.csv', names=['path'])

train_df['full_path'] = 'MURA_DATASET/' + train_df['path']
valid_df['full_path'] = 'MURA_DATASET/' + valid_df['path']

train_df['label'] = train_df['path'].apply(lambda x: '1' if 'positive' in x else '0')
valid_df['label'] = valid_df['path'].apply(lambda x: '1' if 'positive' in x else '0')

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    horizontal_flip=True
)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='full_path',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

valid_generator = valid_datagen.flow_from_dataframe(
    dataframe=valid_df,
    x_col='full_path',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

checkpoint = ModelCheckpoint('mura.h5', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)

print("Training Shuru Ho Rahi Hai Boss...")
history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=10,
    callbacks=[checkpoint]
)


TensorFlow Version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Found 36808 validated image filenames belonging to 2 classes.
Found 3197 validated image filenames belonging to 2 classes.
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Training Shuru Ho Rahi Hai Boss...
Epoch 1/10
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 0s 518ms/step - accuracy: 0.6638 - loss: 0.6162
Epoch 1: val_accuracy improved from None to 0.69252, saving model to mura.h5



Epoch 1: finished saving model to mura.h5
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 667s 554ms/step - accuracy: 0.6869 - loss: 0.5930 - val_accuracy: 0.6925 - val_loss: 0.5828
Epoch 2/10
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 0s 501ms/step - accuracy: 0.7104 - loss: 0.5663
Epoch 2: val_accuracy did not improve from 0.69252
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 591s 514ms/step - accuracy: 0.7125 - loss: 0.5639 - val_accuracy: 0.6703 - val_loss: 0.6321
Epoch 3/10
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 0s 502ms/step - accuracy: 0.7221 - loss: 0.5511
Epoch 3: val_accuracy did not improve from 0.69252
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 622s 514ms/step - accuracy: 0.7230 - loss: 0.5494 - val_accuracy: 0.6916 - val_loss: 0.5944
Epoch 4/10
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step - accuracy: 0.7306 - loss: 0.5413
Epoch 4: val_accuracy improved from 0.69252 to 0.71692, saving model to mura.h5



Epoch 4: finished saving model to mura.h5
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 569s 494ms/step - accuracy: 0.7302 - loss: 0.5403 - val_accuracy: 0.7169 - val_loss: 0.5437
Epoch 5/10
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 0s 484ms/step - accuracy: 0.7352 - loss: 0.5340
Epoch 5: val_accuracy improved from 0.71692 to 0.71880, saving model to mura.h5



Epoch 5: finished saving model to mura.h5
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 571s 496ms/step - accuracy: 0.7358 - loss: 0.5327 - val_accuracy: 0.7188 - val_loss: 0.5401
Epoch 6/10
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 0s 482ms/step - accuracy: 0.7430 - loss: 0.5229
Epoch 6: val_accuracy did not improve from 0.71880
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 576s 500ms/step - accuracy: 0.7398 - loss: 0.5266 - val_accuracy: 0.7125 - val_loss: 0.5543
Epoch 7/10
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 0s 492ms/step - accuracy: 0.7465 - loss: 0.5203
Epoch 7: val_accuracy did not improve from 0.71880
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 581s 505ms/step - accuracy: 0.7461 - loss: 0.5203 - val_accuracy: 0.7047 - val_loss: 0.5669
Epoch 8/10
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 0s 491ms/step - accuracy: 0.7458 - loss: 0.5181
Epoch 8: val_accuracy improved from 0.71880 to 0.72631, saving model to mura.h5



Epoch 8: finished saving model to mura.h5
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 580s 504ms/step - accuracy: 0.7464 - loss: 0.5178 - val_accuracy: 0.7263 - val_loss: 0.5427
Epoch 9/10
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 0s 494ms/step - accuracy: 0.7515 - loss: 0.5127
Epoch 9: val_accuracy improved from 0.72631 to 0.72818, saving model to mura.h5



Epoch 9: finished saving model to mura.h5
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 584s 507ms/step - accuracy: 0.7513 - loss: 0.5122 - val_accuracy: 0.7282 - val_loss: 0.5350
Epoch 10/10
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 0s 493ms/step - accuracy: 0.7494 - loss: 0.5149
Epoch 10: val_accuracy did not improve from 0.72818
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 581s 505ms/step - accuracy: 0.7519 - loss: 0.5119 - val_accuracy: 0.7238 - val_loss: 0.5473
